# Robocup@Home


In [2]:
# Downloading modules that are not in google colab
!pip install -q \
yt-dlp \
ffmpeg \
ultralytics \
imagehash \
roboflow

In [ ]:
# Optional modules
!pip install -q \
supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 10.3 MB/s eta 0:00:00


In [ ]:
# Defining the roboflow values
API_PRIVATE_KEY = "YOUR_API_KEY"
WORKSPACE_ID = "YOUR_WORKSPACE_ID"
PROJECT_ID = "YOUR_PROJECT_ID"
dataset_version = 13

## Downloading videoes and extracting images

The following block downloads video stream using yt-dlp, and extracts frames with ffmpeg.

In [ ]:
import subprocess
from pathlib import Path
from PIL import Image
import imagehash
import numpy as np
import time

# Note, the expected quality of video should be 1080p, and the output quality is qHD
FPS = 1
WIDTH = 960
HEIGHT = 540
PHASH_THRESHOLD = 12 # Increase to get higher difference

def run(url, video_index, destination):
  video_name = f"video{video_index:03d}"
  CURRENT_IMAGE_PATH = destination / video_name
  CURRENT_IMAGE_PATH.mkdir(parents=True, exist_ok=True)

  print(f"Processing {video_name}:")

  # Downloading video as frames
  cmd = f'''
yt-dlp -f "bv*[ext=mp4]/b" -o - "{url}" |
ffmpeg -loglevel error -i pipe:0 -vf "fps={FPS},scale={WIDTH}:{HEIGHT}" -f rawvideo -pix_fmt rgb24 -
'''
  pipe = subprocess.Popen(cmd, stdout=subprocess.PIPE, shell = True)

  frame_size = WIDTH * HEIGHT * 3

  previous_hash = None
  total_frames = 0
  kept_frames = 0

  # Going through all the pictures
  while True:
    raw = pipe.stdout.read(frame_size)
    if not raw:
      break

    # Convert image to array
    frame = np.frombuffer(raw, np.uint8).reshape((HEIGHT, WIDTH, 3))
    image = Image.fromarray(frame)

    current_hash = imagehash.phash(image)
    keep = False

    # Keeping frames that has enough difference
    if (previous_hash is None or
        previous_hash - current_hash > PHASH_THRESHOLD):
      keep = True

    if keep:
      kept_frames += 1
      image.save(CURRENT_IMAGE_PATH / f"{video_name}_{kept_frames:05d}.jpg",
                 quality = 95)

      previous_hash = current_hash

    total_frames += 1

    if total_frames % 50 == 0:
      print(f"Processed {total_frames}, kept {kept_frames}")

  print(f"\nDone: total={total_frames}, kept={kept_frames}")


In [ ]:
video_index = 1

In [ ]:
IMAGE_PATH = Path.cwd() / "dataset" / "frames"

In [ ]:
while url:= input("URL: "):
  start = time.perf_counter()
  run(url, video_index, IMAGE_PATH)
  used_time = time.perf_counter() - start
  print(f"Used time: {used_time: .06f} seconds\n")

  video_index += 1

URL: https://www.youtube.com/watch?v=iFvryLA67bQ
Processing video001:
Processed 50, kept 43
Processed 100, kept 83
Processed 150, kept 124

Done: total=176, kept=140
Used time:  127.016036 seconds

URL: https://www.youtube.com/watch?v=4i52oqq_8Ec
Processing video002:

Done: total=29, kept=25
Used time:  10.432349 seconds

URL: https://www.youtube.com/watch?v=55_VyS18QV0
Processing video003:

Done: total=13, kept=11
Used time:  4.813604 seconds

URL: https://www.youtube.com/watch?v=CyrZvebfKuo
Processing video004:

Done: total=31, kept=23
Used time:  16.200202 seconds

URL: https://www.youtube.com/watch?v=8cQruSrslSU
Processing video005:
Processed 50, kept 41

Done: total=80, kept=52
Used time:  17.905764 seconds

URL: https://www.youtube.com/watch?v=FGtfqwHwvmA
Processing video006:
Processed 50, kept 37

Done: total=55, kept=38
Used time:  41.046735 seconds

URL: 


### (optional) Zipping files

You can zip the images and download it to your *device*

In [ ]:
selected_images = Path.cwd() / "dataset" / "selected_images"
selected_images.mkdir(parents = True, exist_ok = True)

In [ ]:
folders = [folder for folder in IMAGE_PATH.iterdir()]
# Slice the images you want, or the whole video/directory
moving_files = sorted([file for file in folders[0].iterdir() if file.is_file()])

for files in moving_files:
  destination = selected_images / files.name
  files.rename(destination)

In [ ]:
!zip -r images_archive.zip /content/dataset/selected_images

  adding: content/dataset/selected_images/ (stored 0%)
  adding: content/dataset/selected_images/video001.mp4_00118.jpg (deflated 1%)
  adding: content/dataset/selected_images/video001.mp4_00116.jpg (deflated 1%)
  adding: content/dataset/selected_images/video001.mp4_00115.jpg (deflated 1%)
  adding: content/dataset/selected_images/video001.mp4_00113.jpg (deflated 1%)
  adding: content/dataset/selected_images/video001.mp4_00120.jpg (deflated 1%)
  adding: content/dataset/selected_images/video001.mp4_00119.jpg (deflated 1%)
  adding: content/dataset/selected_images/video001.mp4_00111.jpg (deflated 1%)
  adding: content/dataset/selected_images/video001.mp4_00114.jpg (deflated 1%)
  adding: content/dataset/selected_images/video001.mp4_00112.jpg (deflated 1%)
  adding: content/dataset/selected_images/video001.mp4_00117.jpg (deflated 1%)


In [ ]:
from google.colab import files

files.download('images_archive.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Uploading downloaded data to roboflow

In [ ]:
from roboflow import Roboflow
import cv2 # Required to upload properly
import yaml

workspace = Roboflow(api_key=API_PRIVATE_KEY).workspace(WORKSPACE_ID)
project = workspace.project(PROJECT_ID)

def upload(folder_path, yaml_path = None, annotation_path = None):
  # Turning the classes into a dictionary
  if yaml_path is not None:
    with open(yaml_path, "r") as file:
      data = yaml.safe_load(file)

    class_names = data.get("names", [])

    # Ensuring it's the expected format
    if isinstance(class_names, list):
      class_map = {str(index): name for index, name in enumerate(class_names)}
    else:
      class_map = {str(key): value for key, value in class_names.items()}

  for path in folder_path.iterdir():

    # If it's a folder, and-
    if path.is_dir():

      # it's not YOLO predicted (missing .yaml file) images folder
      if yaml_path is None:
        workspace.upload_dataset(
            str(folder_path), # Dataset path
            PROJECT_ID, # This will either create or get a dataset with the given ID
            num_workers=10,
            project_type="instance-segmentation",
            batch_name=None,
            num_retries=0,
            is_prediction=False #optional, set to True if the dataset is not ground truth and needs approval
          )

      # it's YOLO predicted (has .yaml file) images folder
      else:
        workspace.upload_dataset(
          str(folder_path), # Dataset path
          PROJECT_ID, # This will either create or get a dataset with the given ID
          annotation_labelmap = class_map,
          num_workers=10,
          project_type="instance-segmentation",
          batch_name=None,
          num_retries=0,
          is_prediction=False #optional, set to True if the dataset is not ground truth and needs approval
        )

    # If it's file, and-
    elif path.is_file():
      annotation_file = annotation_path / f"{path.stem}.txt"

      # it has annotations
      if annotation_file.exists():
        project.single_upload(
          image_path = str(path),
          annotation_path = str(annotation_path / f"{path.stem}.txt"),
          annotation_labelmap = class_map,
          is_prediction = True # Need human review or not
        )
      # it has no annotations
      else:
        project.single_upload(
        image_path = str(path),
        is_prediction = True # Need human review or not
      )

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
upload(IMAGE_PATH)

loading Roboflow project...


100%|██████████| 289/289 [00:00<00:00, 463716.09it/s]


Uploading to existing project main-bfeau/kitchen-dataset-lnsnb
[UPLOADED] /content/dataset/frames/video001/video001_00001.jpg (F9cWfXAfiAjneUL33Yuk) [1.2s]
[UPLOADED] /content/dataset/frames/video001/video001_00003.jpg (27eAwHtOhHQE1BFIiceP) [1.2s]
[UPLOADED] /content/dataset/frames/video001/video001_00006.jpg (dwYprFtdodEeZQtzhx68) [1.3s]
[UPLOADED] /content/dataset/frames/video001/video001_00007.jpg (QYheeqGJ6Gl1x1UH1d6A) [1.3s]
[UPLOADED] /content/dataset/frames/video001/video001_00010.jpg (jRiSajJcvMV8dqvtvJ0i) [1.5s]
[UPLOADED] /content/dataset/frames/video001/video001_00008.jpg (CWhqZZQkIyEk03YK2kKq) [1.6s]
[UPLOADED] /content/dataset/frames/video001/video001_00009.jpg (xVQsPL02RMTeQaSkhBgL) [1.6s]
[UPLOADED] /content/dataset/frames/video001/video001_00005.jpg (EHbuJ0Ewx4b9ViYw719a) [1.7s]
[UPLOADED] /content/dataset/frames/video001/video001_00002.jpg (yuU6fihNr4sapxs4xqzu) [1.8s]
[UPLOADED] /content/dataset/frames/video001/video001_00012.jpg (QGpoMHDPBfP5H5PtpLiB) [0.6s]
[UPLOAD

100%|██████████| 289/289 [00:00<00:00, 498418.53it/s]


Uploading to existing project main-bfeau/kitchen-dataset-lnsnb
[DUPLICATE] /content/dataset/frames/video001/video001_00010.jpg (jRiSajJcvMV8dqvtvJ0i) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00006.jpg (dwYprFtdodEeZQtzhx68) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00002.jpg (yuU6fihNr4sapxs4xqzu) [0.6s]
[DUPLICATE] /content/dataset/frames/video001/video001_00005.jpg (EHbuJ0Ewx4b9ViYw719a) [0.6s]
[DUPLICATE] /content/dataset/frames/video001/video001_00007.jpg (QYheeqGJ6Gl1x1UH1d6A) [0.6s]
[DUPLICATE] /content/dataset/frames/video001/video001_00003.jpg (27eAwHtOhHQE1BFIiceP) [0.7s]
[DUPLICATE] /content/dataset/frames/video001/video001_00009.jpg (xVQsPL02RMTeQaSkhBgL) [0.6s]
[DUPLICATE] /content/dataset/frames/video001/video001_00004.jpg (u8MLnVCz3RTStwJowpJm) [0.7s]
[DUPLICATE] /content/dataset/frames/video001/video001_00001.jpg (F9cWfXAfiAjneUL33Yuk) [0.7s]
[DUPLICATE] /content/dataset/frames/video001/video001_00011.jpg (QOHjsbWYDaXqhKmYqRaG) [0.4

...

100%|██████████| 289/289 [00:00<00:00, 288676.79it/s]


Uploading to existing project main-bfeau/kitchen-dataset-lnsnb
[DUPLICATE] /content/dataset/frames/video001/video001_00006.jpg (dwYprFtdodEeZQtzhx68) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00003.jpg (27eAwHtOhHQE1BFIiceP) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00008.jpg (CWhqZZQkIyEk03YK2kKq) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00004.jpg (u8MLnVCz3RTStwJowpJm) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00010.jpg (jRiSajJcvMV8dqvtvJ0i) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00005.jpg (EHbuJ0Ewx4b9ViYw719a) [0.6s]
[DUPLICATE] /content/dataset/frames/video001/video001_00009.jpg (xVQsPL02RMTeQaSkhBgL) [0.6s]
[DUPLICATE] /content/dataset/frames/video001/video001_00013.jpg (VrQp64qIBDgtpW1pBPDV) [0.3s]
[DUPLICATE] /content/dataset/frames/video001/video001_00011.jpg (QOHjsbWYDaXqhKmYqRaG) [0.4s]
[DUPLICATE] /content/dataset/frames/video001/video001_00012.jpg (QGpoMHDPBfP5H5PtpLiB) [0.4

100%|██████████| 289/289 [00:00<00:00, 455012.71it/s]

Uploading to existing project main-bfeau/kitchen-dataset-lnsnb


[DUPLICATE] /content/dataset/frames/video001/video001_00010.jpg (jRiSajJcvMV8dqvtvJ0i) [0.4s]
[DUPLICATE] /content/dataset/frames/video001/video001_00002.jpg (yuU6fihNr4sapxs4xqzu) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00005.jpg (EHbuJ0Ewx4b9ViYw719a) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00006.jpg (dwYprFtdodEeZQtzhx68) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00001.jpg (F9cWfXAfiAjneUL33Yuk) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00008.jpg (CWhqZZQkIyEk03YK2kKq) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00009.jpg (xVQsPL02RMTeQaSkhBgL) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00003.jpg (27eAwHtOhHQE1BFIiceP) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00007.jpg (QYheeqGJ6Gl1x1UH1d6A) [0.5s]
[DUPLICATE] /content/dataset/frames/video001/video001_00004.jpg (u8MLnVCz3RTStwJowpJm) [0.6s]
[DUPLICATE] /content/dataset/frames/video001/video001_00011.

## Downloading from YCB benchmark

In [ ]:
file_types = ["berkeley_rgb_highres", "berkeley_rgbd"] # (1) High quality (2) Angle diversity

In [ ]:
YCB_DIRECTORY = Path.cwd() / "dataset" / "YCB-dataset"
YCB_DIRECTORY.mkdir(parents=True, exist_ok=True)

In [ ]:
object_classes = [
  "001_chips_can", "002_master_chef_can", "003_cracker_box", "004_sugar_box", # 0-3
  "005_tomato_soup_can", "006_mustard_bottle", "007_tuna_fish_can", "008_pudding_box", # 4-7
  "009_gelatin_box", "010_potted_meat_can", "011_banana", "012_strawberry", # 8-11
  "013_apple", "014_lemon", "015_peach", "016_pear", # 12-15
  "017_orange", "018_plum", "019_pitcher_base", "021_bleach_cleanser", # 16-19
  "022_windex_bottle", "023_wine_glass", "024_bowl", "025_mug",  # 20-23
  "026_sponge", "027-skillet", "028_skillet_lid", "029_plate",  # 24-27
  "030_fork", "031_spoon", "032_knife", "033_spatula", "035_power_drill", # 28-31
  "036_wood_block", "037_scissors", "038_padlock", "039_key_set", # 32-35
  "040_large_marker", "041_small_marker", "042_adjustable_wrench", "043_phillips_screwdriver", # 36-39
  "044_flat_screwdriver", "048_hammer", "050_medium_clamp", "051_large_clamp", # 40-43
  "052_extra_large_clamp", "053_mini_soccer_ball", "054_softball", "055_baseball", # 44-47
  "056_tennis_ball", "057_racquetball", "058_golf_ball", "059_chain", # 48-51
  "061_foam_brick", "062_dice", "063_marbles", "065-a_cups", # 52-55
  "065-b_cups", "065-c_cups", "065-d_cups", "065-e_cups", # 56-59
  "065-f_cups", "065-g_cups", "065-h_cups", "065-i_cups", # 60-63
  "065-j_cups", "070-a_colored_wood_blocks", "070-b_colored_wood_blocks", "071_nine_hole_peg_test", # 64-67
  "072-a_toy_airplane", "072-b_toy_airplane", "072-c_toy_airplane", "072-d_toy_airplane", # 68-71
  "072-e_toy_airplane", "073-a_lego_duplo", "073-b_lego_duplo", "073-c_lego_duplo", # 72-75
  "073-d_lego_duplo", "073-e_lego_duplo", "073-f_lego_duplo", "073-g_lego_duplo", # 76-79
  "077_rubiks_cube" # 80
]

In [ ]:
robocup_objects = [
    "001_chips_can", #0
    "002_master_chef_can",
    "003_cracker_box",
    "004_sugar_box",
    "005_tomato_soup_can",
    "006_mustard_bottle",
    "007_tuna_fish_can",
    "008_pudding_box",
    "009_gelatin_box",
    "010_potted_meat_can",
    "011_banana",
    "012_strawberry",
    "013_apple",
    "014_lemon",
    "015_peach",
    "016_pear",
    "017_orange",
    "018_plum", #17
    "019_pitcher_base",
    "021_bleach_cleanser",
    "022_windex_bottle",
    "024_bowl",
    "025_mug",
    "026_sponge",
    "029_plate",
    "030_fork",
    "031_spoon",
    "032_knife",
    "033_spatula",
    "035_power_drill",
    "037_scissors",
    "040_large_marker",
    "042_adjustable_wrench",
    "048_hammer",
    "077_rubiks_cube" #34
]

len(robocup_objects)

35

In [ ]:
def get_objects(*index, overview = object_classes):
  return [overview[i] for i in index]

# One object class takes approximately 26 mins with CPU
objects = get_objects(0, 2，
                      overview = robocup_objects)

SyntaxError: invalid character '，' (U+FF0C) (578568569.py, line 5)

In [ ]:
import urllib.request
import tarfile
from pathlib import Path

base_url = "http://ycb-benchmarks.s3-website-us-east-1.amazonaws.com/data/berkeley"

for obj in robocup_objects:
  url = f"{base_url}/{obj}/{obj}_{file_types[1]}.tgz"
  temporary_path = YCB_DIRECTORY / f"{obj}_{file_types[0]}.tgz"

  urllib.request.urlretrieve(url, temporary_path)

  with tarfile.open(temporary_path, "r:gz") as tar:
    tar.extractall(path = YCB_DIRECTORY)
  temporary_path.unlink()

Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.


Downloading to google Mydrive for further downloading to local pc

In [ ]:
def mask_to_yolo_polygons(mask_path, img_w, img_h, class_id):
    # Read the mask as grayscale
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None: return None

    # If the background is white (mean > 127), invert the picture so that the object is white
    if np.mean(mask) > 127:
      mask = cv2.bitwise_not(mask)

    # Find contour
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    yolo_polygons = []
    for cnt in contours:
      if cv2.contourArea(cnt) < 200: # Ignore small noise
        continue

      points = []
      for p in cnt:
        # Normalize coordinates
        x_norm = p[0][0] / img_w
        y_norm = p[0][1] / img_h
        points.append(f"{x_norm:.6f} {y_norm:.6f}")

      yolo_polygons.append(f"{class_id} {' '.join(points)}")

    return "\n".join(yolo_polygons) if yolo_polygons else None

In [ ]:
import os
import random
import shutil
from pathlib import Path

SOURCE_DIR = YCB_DIRECTORY
OUTPUT_DIR = Path.cwd() / "downloade_to_local"
SPLITS = {'train': 0.7, 'val': 0.2, 'test': 0.1}

for split in SPLITS.keys():
  (OUTPUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
  (OUTPUT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

classes = sorted([d for d in SOURCE_DIR.iterdir() if d.is_dir()])

'chips_can'

In [ ]:
for class_idx, class_folder in enumerate(classes):
  obj_name = class_folder.name[4:]
  print(f"Prosesserer {obj_name} (ID: {class_idx})...")

  # Gather all jpg pictures and shuffle them for randomness
  images = list(class_folder.glob("*.jpg"))
  random.shuffle(images)

  n = len(images)
  train_end = int(n * SPLITS['train'])
  val_end = train_end + int(n * SPLITS['val'])

  # Separete the pictures
  for i, img_path in enumerate(images):
    if i < train_end:
      current_split = 'train'
    elif i < val_end:
      current_split = 'val'
    else:
      current_split = 'test'

    unique_name = f"{obj_name}_{i}"

    dest_img = OUTPUT_DIR / current_split / "images" / unique_name
    shutil.copy(img_path, dest_img)

    mask_path = class_folder / "masks" / f"{img_path.stem}_mask.pbm"

    if mask_path.exists():
      yolo_data = mask_to_yolo_polygons(mask_path, 640, 480, class_idx)

      if yolo_data:
        label_name = f"{Path(unique_name).stem}.txt"
        dest_label = OUTPUT_DIR / current_split / "labels" / label_name

        with open(dest_label, "w") as f:
          f.write(yolo_data)

print(f"\nDatasettet er lagret i: {OUTPUT_DIR.absolute()}")

Prosesserer chips_can (ID: 0)...
Prosesserer master_chef_can (ID: 1)...
Prosesserer cracker_box (ID: 2)...
Prosesserer sugar_box (ID: 3)...
Prosesserer tomato_soup_can (ID: 4)...
Prosesserer mustard_bottle (ID: 5)...
Prosesserer tuna_fish_can (ID: 6)...
Prosesserer pudding_box (ID: 7)...
Prosesserer gelatin_box (ID: 8)...
Prosesserer potted_meat_can (ID: 9)...
Prosesserer banana (ID: 10)...
Prosesserer strawberry (ID: 11)...
Prosesserer apple (ID: 12)...
Prosesserer lemon (ID: 13)...
Prosesserer peach (ID: 14)...
Prosesserer pear (ID: 15)...
Prosesserer orange (ID: 16)...
Prosesserer plum (ID: 17)...
Prosesserer pitcher_base (ID: 18)...
Prosesserer bleach_cleanser (ID: 19)...
Prosesserer windex_bottle (ID: 20)...
Prosesserer bowl (ID: 21)...
Prosesserer sponge (ID: 22)...
Prosesserer plate (ID: 23)...
Prosesserer fork (ID: 24)...
Prosesserer spoon (ID: 25)...
Prosesserer knife (ID: 26)...
Prosesserer spatula (ID: 27)...
Prosesserer power_drill (ID: 28)...
Prosesserer scissors (ID: 29).

In [ ]:
import shutil
import os

drive_path = '/content/drive/MyDrive/YCB_Transfer/'

if not os.path.exists(drive_path):
  os.makedirs(drive_path)

files_to_upload = ['yolo_train.zip', 'yolo_val.zip', 'yolo_test.zip']

for file in files_to_upload:
  if os.path.exists(file):
    print(f"Laster opp {file} til Google Drive...")
    shutil.copy(file, drive_path + file)
    print(f"Ferdig med {file}")
  else:
    print(f"Fant ikke {file}. Sjekk at navnet stemmer.")

print("\n--- Alle filer er nå trygt på Google Drive! ---")

Fant ikke y. Sjekk at navnet stemmer.
Fant ikke o. Sjekk at navnet stemmer.
Fant ikke l. Sjekk at navnet stemmer.
Fant ikke o. Sjekk at navnet stemmer.
Fant ikke _. Sjekk at navnet stemmer.
Fant ikke t. Sjekk at navnet stemmer.
Fant ikke r. Sjekk at navnet stemmer.
Fant ikke a. Sjekk at navnet stemmer.
Fant ikke i. Sjekk at navnet stemmer.
Fant ikke n. Sjekk at navnet stemmer.
Laster opp . til Google Drive...


IsADirectoryError: [Errno 21] Is a directory: '.'

Delete the finished set

In [ ]:
import shutil

folder_path = '/content/dataset/YCB-dataset'
if os.path.exists(folder_path):
  shutil.rmtree(folder_path)
  print(f"Slettet {folder_path}")

Slettet /content/dataset/YCB-dataset


### Using COCO.yaml trained model to predict the images

In [ ]:
from ultralytics import YOLO

model_COCO = YOLO("yolo26s-seg.pt")

model_COCO.train(
    data = f"coco8-seg.yaml", # Expected the .yaml file
    epochs=300,      # High maximum limit
    patience=50,     # Stop early if no improvement for 50 epochs
    imgsz=640,
    batch=16,         # Small batch size is fine for small datasets
    device = 0,
    augment = True # Data augmentation
    )

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [ ]:
best_model_COCO = YOLO("/content/runs/segment/train/weights/best.pt")

In [ ]:
results_COCO = dict()

In [ ]:
prediction_videos = sorted([folder for folder in YCB_DIRECTORY.iterdir()])

In [ ]:
for video in prediction_videos:
  if video.name not in results_COCO:
    print(f"\n\nAdding {video.name}: ")
    results_COCO[video.name] = best_model_COCO.predict(source = video, save = True, show = False)



Adding 013_apple: 

image 1/600 /content/dataset/YCB-dataset/013_apple/N1_0.jpg: 448x640 1 apple, 100.1ms
image 2/600 /content/dataset/YCB-dataset/013_apple/N1_102.jpg: 448x640 1 apple, 18.1ms
image 3/600 /content/dataset/YCB-dataset/013_apple/N1_105.jpg: 448x640 1 apple, 18.4ms
image 4/600 /content/dataset/YCB-dataset/013_apple/N1_108.jpg: 448x640 1 apple, 23.5ms
image 5/600 /content/dataset/YCB-dataset/013_apple/N1_111.jpg: 448x640 1 apple, 21.9ms
image 6/600 /content/dataset/YCB-dataset/013_apple/N1_114.jpg: 448x640 1 apple, 17.5ms
image 7/600 /content/dataset/YCB-dataset/013_apple/N1_117.jpg: 448x640 1 apple, 17.5ms
image 8/600 /content/dataset/YCB-dataset/013_apple/N1_12.jpg: 448x640 1 apple, 17.8ms
image 9/600 /content/dataset/YCB-dataset/013_apple/N1_120.jpg: 448x640 1 apple, 17.5ms
image 10/600 /content/dataset/YCB-dataset/013_apple/N1_123.jpg: 448x640 1 apple, 17.5ms
image 11/600 /content/dataset/YCB-dataset/013_apple/N1_126.jpg: 448x640 1 apple, 17.4ms
image 12/600 /content

## Training model

First, download the dataset from Roboflow

In [ ]:
from roboflow import Roboflow
from ultralytics import YOLO

# Getting the dataset from Roboflow
project = Roboflow(api_key=API_PRIVATE_KEY).workspace(WORKSPACE_ID).project(PROJECT_ID)

dataset = project.version(dataset_version).download("yolo26")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Kitchen-dataset-13 in yolo26:: 100%|██████████| 795/795 [00:00<00:00, 5318.08it/s]


Checking if there are any points less than 3, which causes segmentation model to fail.

In [ ]:
from pathlib import Path

for label in ["train", "valid", "test"]:
  label_dir = Path(f"{dataset.location}/{label}/labels")

  print(f"---\n{label} label has the following error files:\n")
  for file in label_dir.glob('*.txt'):
    # Checking if one of the lines has less than 3 pixels (6 in total for coordinates)
    if any(0 < len(line.split()) <= 6 for line in file.read_text().splitlines()):
      print(f"{file.name}")

---
train label has the following error files:

---
valid label has the following error files:

---
test label has the following error files:



Now it's properly training model

In [ ]:
model = YOLO("yolo26s-seg.pt")

model.train(
    data = f"{dataset.location}/data.yaml", # Expected the .yaml file
    epochs=300,      # High maximum limit
    patience=50,     # Stop early if no improvement for 50 epochs
    imgsz=640,
    batch=16,         # Small batch size is fine for small datasets
    device = 0,
    augment = True # Data augmentation
    )

Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Kitchen-dataset-13/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, per

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f78152dc2f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

Gather traning data

In [ ]:
prediction_video_index = 1

In [ ]:
PREDICTION_IMAGE_PATH = Path.cwd() / "dataset" / "predicted_frames"

In [ ]:
while url:= input("URL: "):
  start = time.perf_counter()
  run(url, prediction_video_index, PREDICTION_IMAGE_PATH)
  used_time = time.perf_counter() - start
  print(f"Used time: {used_time: .06f} seconds\n")

  prediction_video_index += 1

URL: https://www.youtube.com/watch?v=U8tE6K4dy1A
Processing video001:
Processed 50, kept 37
Processed 100, kept 76

Done: total=120, kept=96
Used time:  38.439825 seconds

URL: 


In [ ]:
best_model = YOLO("/content/runs/segment/train/weights/best.pt")

In [ ]:
results = dict()

In [ ]:
prediction_videos = sorted([folder for folder in PREDICTION_IMAGE_PATH.iterdir()])

In [ ]:
for video in prediction_videos:
  if video.name not in results:
    print(f"\n\nAdding {video.name}: ")
    results[video.name] = best_model.predict(source = video, save = True, show = False)



Adding video001: 

image 1/96 /content/dataset/predicted_frames/video001/video001_00001.jpg: 384x640 (no detections), 70.7ms
image 2/96 /content/dataset/predicted_frames/video001/video001_00002.jpg: 384x640 (no detections), 15.4ms
image 3/96 /content/dataset/predicted_frames/video001/video001_00003.jpg: 384x640 (no detections), 15.4ms
image 4/96 /content/dataset/predicted_frames/video001/video001_00004.jpg: 384x640 (no detections), 15.3ms
image 5/96 /content/dataset/predicted_frames/video001/video001_00005.jpg: 384x640 (no detections), 15.3ms
image 6/96 /content/dataset/predicted_frames/video001/video001_00006.jpg: 384x640 (no detections), 15.4ms
image 7/96 /content/dataset/predicted_frames/video001/video001_00007.jpg: 384x640 (no detections), 15.4ms
image 8/96 /content/dataset/predicted_frames/video001/video001_00008.jpg: 384x640 (no detections), 13.2ms
image 9/96 /content/dataset/predicted_frames/video001/video001_00009.jpg: 384x640 1 football, 13.0ms
image 10/96 /content/dataset/p

Manually inspecting the predicted images

In [ ]:
REVIEW_IMAGE_PATH = Path.cwd() / "dataset" / "upload_frames"
REVIEW_IMAGE_PATH.mkdir(parents = True, exist_ok = True)
REVIEW_ANNOTATION_PATH = Path.cwd() / "dataset" / "upload_annotations"
REVIEW_ANNOTATION_PATH.mkdir(parents = True, exist_ok = True)

In [ ]:
import matplotlib.pyplot as plt
import cv2

def reviewing_YOLO_prediction(results, accuracy_threshold = 0.70,
                              human_review = True, include_none = True):
  image_index, total_images = 1, len(results)

  for result in results:
    # To select images that has not been predictor or under a certain acccuracy
    confidences = result.boxes.conf.tolist()

    selected = False

    if include_none and not confidences: # If it has no annotations
      selected = True
    else:
      for score in confidences: # If one of the prediction accuracy is lower than accuracy_threshold
        if round(score, 2) < accuracy_threshold:
          selected = True

    if selected:
      # The pictures will be manually selected
      if human_review:
        plt.ion()
        fig = plt.figure(figsize=(10, 6))

        # YOLO images are in BGR format, so we need to convert the color dimension (3-dimension) in reverse
        img_rgb = result.plot()[:, :, ::-1]

        plt.imshow(img_rgb)
        plt.axis('off')

        print(f"\n{image_index}/{total_images}")

        plt.show()
        plt.pause(0.1)

        user_choice = input("Skip (0/None)\nKeep (1)\nStop (2): ")

        if user_choice == "1":
          base_name = f"predicted_{Path(result.path).stem}"
          img_path = REVIEW_IMAGE_PATH / f"{base_name}.jpg"
          annotation_path = REVIEW_ANNOTATION_PATH / f"{base_name}.txt"

          # Opencv (cv2) has images as BGR in default/standard, so we need to convert it
          raw_rgb = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB)
          rgb_file = Image.fromarray(raw_rgb)
          rgb_file.save(img_path)

          result.save_txt(str(annotation_path))

        elif user_choice == "2":
          plt.close(fig)
          break

        plt.close(fig)

      # The pictures are automatically selected based on the accuracy_threshold
      else:
        base_name = f"predicted_{Path(result.path).stem}"
        img_path = REVIEW_IMAGE_PATH / f"{base_name}.jpg"
        annotation_path = REVIEW_ANNOTATION_PATH / f"{base_name}.txt"

        # Opencv (cv2) has images as BGR in default/standard, so we need to convert it
        raw_rgb = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB)
        rgb_file = Image.fromarray(raw_rgb)
        rgb_file.save(img_path)

        result.save_txt(str(annotation_path))

    image_index += 1

  plt.ioff()

In [ ]:
for name, result in results.items():
  user_input = input(f"\nSkip (0/None)\nReview {name} (1)\ndone reviewing (2):")

  if user_input == "0":
    continue
  elif user_input == "1":
    reviewing_YOLO_prediction(result)
  elif user_input == "2":
    break

Output hidden; open in https://colab.research.google.com to view.

### Upload the annotated images to roboflow

In [ ]:
yaml_file = Path(f"{dataset.location}/data.yaml")

In [ ]:
upload(REVIEW_IMAGE_PATH, yaml_file, annotation_path = REVIEW_ANNOTATION_PATH)